# intelliCARE Music Play-a-Long: beatmap pipeline

Song: **翎羽悸動 | Fluttering Feathers, Stirring Hearts**, performed by **Selina Goh 吴宛颖**

Pipeline: MP4 → ffmpeg WAV → **Demucs htdemucs** drum stem → **ADTOF** drum transcription (falls back to band-split onset detection) → **librosa** beats, downbeats and sections → `beatmap.json` + `peaks.json` + poster frame.

1. Runtime → Change runtime type → **GPU**.
2. Runtime → **Run all**.
3. The last cell downloads `beatmap.json`. Open the app with `?editor=1`, Import it, fix it, then use **Copy inline** and paste the block into `index.html`.

In [ ]:
#@title 1 · Setup and download
!pip -q install demucs librosa pretty_midi soundfile
!apt -qq install -y ffmpeg > /dev/null
VIDEO_URL = "https://d2yy42b28nebea.cloudfront.net/activity/9_S9_-_D3_MASTER_20260311_c64c9af3_20260311_6f90eb13.mp4"
SONG = dict(title_zh="翎羽悸動", title_en="Fluttering Feathers, Stirring Hearts", artist="Selina Goh 吴宛颖", video=VIDEO_URL)
!wget -q -O song.mp4 "$VIDEO_URL"
!ffmpeg -y -loglevel error -i song.mp4 -vn -ac 2 -ar 44100 song.wav
!ffmpeg -y -loglevel error -i song.mp4 -vn -ac 1 -ar 44100 song_mono.wav
import soundfile as sf
info = sf.info('song.wav'); DURATION = info.duration
print('duration (s):', round(DURATION, 2))

In [ ]:
#@title 2 · Isolate the drums with Demucs (htdemucs)
!python -m demucs -n htdemucs --two-stems=drums song.wav -o sep
DRUMS = 'sep/htdemucs/song/drums.wav'
import os; print('drum stem exists:', os.path.exists(DRUMS))

In [ ]:
#@title 3 · PATH A: ADTOF transcription → drums.mid (optional, skipped automatically on failure)
# ADTOF's API changes between versions. If this cell fails, Path B (next cell) is used instead.
# See https://github.com/MZehren/ADTOF (bin/drumTranscriptor) for the current invocation.
ADTOF_OK = False
try:
    !pip -q install git+https://github.com/MZehren/ADTOF.git
    import glob, subprocess
    os.makedirs('adtof_out', exist_ok=True)
    r = subprocess.run(['drumTranscriptor', DRUMS, 'adtof_out'], capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    mids = glob.glob('adtof_out/**/*.mid', recursive=True)
    if mids:
        os.replace(mids[0], 'drums.mid'); ADTOF_OK = True
except Exception as e:
    print('ADTOF unavailable:', e)
print('ADTOF_OK =', ADTOF_OK)

In [ ]:
#@title 4 · PATH B (fallback): band-split onset detection on the drum stem
import numpy as np, librosa, scipy.signal as sps
y, sr = librosa.load(DRUMS, sr=44100, mono=True)
HOP = 256
def band(lo, hi):
    if lo is None: sos = sps.butter(4, hi, btype='low', fs=sr, output='sos')
    elif hi is None: sos = sps.butter(4, lo, btype='high', fs=sr, output='sos')
    else: sos = sps.butter(4, [lo, hi], btype='band', fs=sr, output='sos')
    return sps.sosfiltfilt(sos, y)
def onsets(yb, delta, wait_s):
    env = librosa.onset.onset_strength(y=yb, sr=sr, hop_length=HOP)
    fr = librosa.onset.onset_detect(onset_envelope=env, sr=sr, hop_length=HOP, backtrack=False, delta=delta, wait=max(1, int(wait_s * sr / HOP)))
    return librosa.frames_to_time(fr, sr=sr, hop_length=HOP), env[fr] / (env.max() + 1e-9)
# Tune delta per song if the chart is too dense or too sparse.
K = onsets(band(None, 150), 0.08, 0.12)
S = onsets(band(180, 2500), 0.10, 0.12)
H = onsets(band(7000, 16000), 0.06, 0.07)
# cymbal = high-band peak whose energy stays above 50 % for > 400 ms
hb = np.abs(librosa.stft(band(5000, None), hop_length=HOP)).sum(0)
hb = hb / (hb.max() + 1e-9)
def long_decay(t):
    f = int(t * sr / HOP); f2 = f + int(0.4 * sr / HOP)
    return f2 < len(hb) and hb[f2] > 0.5 * hb[f]
fb = []
for t, v in zip(*K): fb.append(dict(t=round(float(t), 3), l='kick', v=round(float(min(1, .4 + v)), 2), bud=False))
for t, v in zip(*S): fb.append(dict(t=round(float(t), 3), l='snare', v=round(float(min(1, .4 + v)), 2), bud=False))
for t, v in zip(*H): fb.append(dict(t=round(float(t), 3), l='cymbal' if long_decay(t) else 'hihat', v=round(float(min(1, .4 + v)), 2), bud=False))
print('fallback notes:', len(fb))

In [ ]:
#@title 5 · Tempo, beats, downbeats, sections
y_mix, sr2 = librosa.load('song_mono.wav', sr=22050, mono=True)
tempo, beat_frames = librosa.beat.beat_track(y=y_mix, sr=sr2, units='frames')
beats = librosa.frames_to_time(beat_frames, sr=sr2)
BPM = float(np.atleast_1d(tempo)[0])
# downbeat phase: choose the offset (0..3) whose beats carry the most kick energy
kick_env = librosa.onset.onset_strength(y=band(None, 150), sr=sr, hop_length=HOP)
kt = librosa.frames_to_time(np.arange(len(kick_env)), sr=sr, hop_length=HOP)
kb = np.interp(beats, kt, kick_env)
phase = int(np.argmax([kb[p::4].mean() for p in range(4)]))
downbeats = beats[phase::4]
# sections: agglomerative segmentation on beat-synchronous chroma + MFCC
chroma = librosa.feature.chroma_cqt(y=y_mix, sr=sr2)
mfcc = librosa.feature.mfcc(y=y_mix, sr=sr2, n_mfcc=13)
feat = librosa.util.sync(np.vstack([chroma, librosa.util.normalize(mfcc, axis=1)]), beat_frames)
k = int(np.clip(round(DURATION / 25), 4, 10))
bounds = librosa.segment.agglomerative(feat, k)
bt = librosa.frames_to_time(librosa.util.fix_frames(beat_frames)[bounds], sr=sr2)
names = ['Intro', 'Verse 1', 'Chorus', 'Verse 2', 'Chorus', 'Bridge', 'Chorus', 'Outro', 'Section 9', 'Section 10']
sections = [dict(t=round(float(t), 2), name=names[i] if i < len(names) else 'Section %d' % (i + 1)) for i, t in enumerate(sorted(set(np.round(bt, 2))))]
if not sections or sections[0]['t'] > 0.5: sections.insert(0, dict(t=0.0, name='Intro'))
print('BPM %.1f, %d beats, %d bars, sections:' % (BPM, len(beats), len(downbeats)), sections)
print('Rename the sections in the Editor after listening.')

In [ ]:
#@title 6 · MIDI / fallback → lanes → beatmap.json (+ peaks.json, poster)
import json, pretty_midi, base64, datetime
GM = {**{p: 'kick' for p in (35, 36)}, **{p: 'snare' for p in (37, 38, 39, 40)},
      **{p: 'tom' for p in (41, 43, 45, 47, 48, 50)}, **{p: 'hihat' for p in (42, 44, 46)},
      **{p: 'cymbal' for p in (49, 51, 52, 53, 55, 57, 59)}}
if ADTOF_OK:
    pm = pretty_midi.PrettyMIDI('drums.mid')
    notes = [dict(t=round(n.start, 3), l=GM[n.pitch], v=round(n.velocity / 127, 2), bud=False)
             for inst in pm.instruments for n in inst.notes if n.pitch in GM]
    generator = 'demucs-htdemucs + ADTOF + librosa'
else:
    notes = fb; generator = 'demucs-htdemucs + band-split onsets (fallback) + librosa'
notes.sort(key=lambda n: n['t'])
# de-duplicate: same lane within 30 ms -> keep first
last, clean = {}, []
for n in notes:
    if n['l'] in last and n['t'] - last[n['l']] < 0.030: continue
    last[n['l']] = n['t']; clean.append(n)
beatmap = dict(version=1,
    song=dict(**SONG, duration=round(DURATION, 2), previewStart=30.0),
    offsetMs=0, bpm=round(BPM, 2),
    beats=[round(float(b), 3) for b in beats], downbeats=[round(float(b), 3) for b in downbeats],
    sections=sections, lanes=['kick', 'snare', 'hihat', 'tom', 'cymbal'],
    charts=dict(full=clean, fire=[], groove=[], gentle=[]),
    meta=dict(generator=generator, created=datetime.date.today().isoformat()))
json.dump(beatmap, open('beatmap.json', 'w', encoding='utf-8'), ensure_ascii=False)
# drum-stem waveform peaks at 100 per second (for the Editor)
win = sr // 100
pk = [round(float(np.abs(y[i:i + win]).max()), 3) for i in range(0, len(y), win)]
json.dump(dict(rate=100, peaks=pk), open('peaks.json', 'w'))
# poster frame (base64 JPEG, about 40 KB or less)
!ffmpeg -y -loglevel error -ss 30 -i song.mp4 -frames:v 1 -vf scale=480:-1 -q:v 7 poster.jpg
print('poster base64 length:', len(base64.b64encode(open('poster.jpg', 'rb').read())))
from collections import Counter
print(len(clean), 'notes', Counter(n['l'] for n in clean))

In [ ]:
#@title 7 · Download
from google.colab import files
for f in ['beatmap.json', 'peaks.json', 'poster.jpg']: files.download(f)

In [ ]:
#@title 8 · Web-optimised video (fixes mid-song buffering)
# The master MP4 is ~524 MB (~18 Mbit/s), too heavy for phones on Wi-Fi/4G.
# This makes a 720p H.264 copy at ~3.5 Mbit/s with 'faststart' (index at the front) -> roughly 100 MB.
# Upload song_web.mp4 next to the original on S3/CloudFront, then set Config.VIDEO_WEB in index.html to its URL.
# Optional (enables 'Save video to this device'): add a CORS rule on the S3 bucket allowing GET/HEAD from your GitHub Pages origin.
!ffmpeg -y -loglevel error -i song.mp4 -vf "scale='if(gt(iw,ih),-2,720)':'if(gt(iw,ih),720,-2)'" -c:v libx264 -preset slow -crf 23 -maxrate 3500k -bufsize 7000k -profile:v high -pix_fmt yuv420p -c:a aac -b:a 160k -movflags +faststart song_web.mp4
!ls -lh song_web.mp4
files.download('song_web.mp4')